# Lab 6 Tasks
### Name: Gul Fatima
### Student ID: 20241-35943

## Task 1 - Map Coloring
### Task 1.1: Backtracking

In [83]:

variables = ["A", "B", "C", "D", "E", "F", "G"]
colors = ["Red", "Green", "Blue"]

domains = {v: colors.copy() for v in variables}

neighbors = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D", "E"],
    "D": ["B", "C", "E", "F"],
    "E": ["C", "D", "F", "G"],
    "F": ["D", "E", "G"],
    "G": ["E", "F"]
}

def is_consistent(variable, color, assignment):
    for neighbor in neighbors[variable]:
        if neighbor in assignment:
            if assignment[neighbor] == color:
                return False
    return True

def backtracking_search(assignment):
    if len(assignment) == len(variables):
        return assignment.copy()

    variable = next(v for v in variables if v not in assignment)

    for color in domains[variable]:
        if is_consistent(variable, color, assignment):
            assignment[variable] = color

            result = backtracking_search(assignment)
            if result is not None:
                return result

            del assignment[variable]

    return None

solution = backtracking_search({})

if solution:
    print("Valid Map Coloring:")
    for region, color in solution.items():
        print(f"{region}: {color}")
else:
    print("No solution exists.")


Valid Map Coloring:
A: Red
B: Green
C: Blue
D: Red
E: Green
F: Blue
G: Red


### Task 1.2: Forward Checking

In [84]:

attempted_assignments = 0
backtracks = 0

def forward_checking(assignment, remaining_domains):
    global attempted_assignments, backtracks

    if len(assignment) == len(variables):
        return assignment.copy()

    variable = next(v for v in variables if v not in assignment)

    for color in remaining_domains[variable]:
        attempted_assignments += 1

        if not is_consistent(variable, color, assignment):
            continue

        assignment[variable] = color

        new_domains = {
            v: values.copy()
            for v, values in remaining_domains.items()
        }

        new_domains[variable] = [color]
        valid = True

        for neighbor in neighbors[variable]:
            if neighbor not in assignment:
                if color in new_domains[neighbor]:
                    new_domains[neighbor].remove(color)

                if not new_domains[neighbor]:
                    valid = False
                    break

        if valid:
            result = forward_checking(assignment, new_domains)

            if result is not None:
                return result

        del assignment[variable]

    backtracks += 1
    return None

solution = forward_checking({}, domains)

print("Forward Checking Solution:\n", solution)
print("Attempted Assignments:", attempted_assignments)
print("Backtracks:", backtracks)


Forward Checking Solution:
 {'A': 'Red', 'B': 'Green', 'C': 'Blue', 'D': 'Red', 'E': 'Green', 'F': 'Blue', 'G': 'Red'}
Attempted Assignments: 7
Backtracks: 0


### Task 1.3: AC-3


In [85]:
from collections import deque

def revise(domains, xi, xj):
    revised = False

    for color in domains[xi][:]:
        if not any(color != other for other in domains[xj]):
            domains[xi].remove(color)
            revised = True

    return revised

def ac3(domains):
    queue = deque(
        (xi, xj)
        for xi in variables
        for xj in neighbors[xi]
    )

    while queue:
        xi, xj = queue.popleft()

        if revise(domains, xi, xj):
            if not domains[xi]:
                return False

            for xk in neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))

    return True

ac3_domains = {
    v: values.copy() for v, values in domains.items()
}

print("Domains Before AC-3:")
for region, colors in domains.items():
    print(f"{region}: {colors}")

result = ac3(ac3_domains)

print("\nAC-3 Successful:", result)

print("\nDomains After AC-3:")
for region, colors in ac3_domains.items():
    print(f"{region}: {colors}")


Domains Before AC-3:
A: ['Red', 'Green', 'Blue']
B: ['Red', 'Green', 'Blue']
C: ['Red', 'Green', 'Blue']
D: ['Red', 'Green', 'Blue']
E: ['Red', 'Green', 'Blue']
F: ['Red', 'Green', 'Blue']
G: ['Red', 'Green', 'Blue']

AC-3 Successful: True

Domains After AC-3:
A: ['Red', 'Green', 'Blue']
B: ['Red', 'Green', 'Blue']
C: ['Red', 'Green', 'Blue']
D: ['Red', 'Green', 'Blue']
E: ['Red', 'Green', 'Blue']
F: ['Red', 'Green', 'Blue']
G: ['Red', 'Green', 'Blue']


## Task 2 - Sudoku Solver


In [86]:
puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

In [87]:
def print_grid(grid):
    for row in grid:
        print(" ".join(str(num) for num in row))

In [88]:
def is_valid(grid, row, col, num):
    # Check row
    if num in grid[row]:
        return False

    # Check column
    for r in range(9):
        if grid[r][col] == num:
            return False

    # Check 3x3 subgrid
    start_row = (row // 3) * 3
    start_col = (col // 3) * 3

    for r in range(start_row, start_row + 3):
        for c in range(start_col, start_col + 3):
            if grid[r][c] == num:
                return False

    return True


In [89]:
def find_empty(grid):
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0:
                return r, c
    return None

### Backtracking Solution

In [90]:
def backtracking(grid, stats):
    empty = find_empty(grid)

    if empty is None:
        return True

    row, col = empty

    for num in range(1, 10):
        stats["attempts"] += 1

        if is_valid(grid, row, col, num):
            grid[row][col] = num

            if backtracking(grid, stats):
                return True

            grid[row][col] = 0

    stats["backtracks"] += 1
    return False

In [91]:
grid1 = [row.copy() for row in puzzle]
bt_stats = {"attempts": 0, "backtracks": 0}
bt_solved = backtracking(grid1, bt_stats)

print("\nBacktracking Solution:")
if bt_solved:
    print_grid(grid1)
else:
    print("No solution exists.")


Backtracking Solution:
5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9


### Forward Checking Solution

In [92]:
def get_domain(grid, row, col):
    return [
        num for num in range(1, 10)
        if is_valid(grid, row, col, num)
    ]


def forward_checking(grid, stats):
    empty = find_empty(grid)

    if empty is None:
        return True

    row, col = empty

    for num in get_domain(grid, row, col):
        stats["attempts"] += 1
        grid[row][col] = num

        valid = True

        # Check remaining empty cells
        for r in range(9):
            for c in range(9):
                if grid[r][c] == 0 and not get_domain(grid, r, c):
                    valid = False
                    break
            if not valid:
                break

        if valid and forward_checking(grid, stats):
            return True

        grid[row][col] = 0

    stats["backtracks"] += 1
    return False


In [93]:
grid2 = [row.copy() for row in puzzle]
fc_stats = {"attempts": 0, "backtracks": 0}
fc_solved = forward_checking(grid2, fc_stats)

print("\nForward Checking Solution:")
if fc_solved:
    print_grid(grid2)
else:
    print("No solution exists.")


Forward Checking Solution:
5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9


### Overall Statistics

In [94]:
print("\nSearch Statistics:")
print("Backtracking Attempts:", bt_stats["attempts"])
print("Backtracking Backtracks:", bt_stats["backtracks"])
print("Forward Checking Attempts:", fc_stats["attempts"])
print("Forward Checking Backtracks:", fc_stats["backtracks"])


Search Statistics:
Backtracking Attempts: 37652
Backtracking Backtracks: 4157
Forward Checking Attempts: 516
Forward Checking Backtracks: 282


## Task 2 - University Examination Scheduling

In [95]:
exams = ["AI", "DB", "OS", "CN", "PF"]

slots = [
    "Monday 9 AM",
    "Monday 11 AM",
    "Monday 2 PM",
    "Tuesday 9 AM"
]

domains = {exam: slots.copy() for exam in exams}

conflicts = {
    "AI": ["DB", "OS"],
    "DB": ["AI", "CN"],
    "OS": ["AI", "CN", "PF"],
    "CN": ["DB", "OS", "PF"],
    "PF": ["OS", "CN"]
}


In [96]:
def is_consistent(exam, slot, assignment):
    for other in conflicts[exam]:
        if other in assignment and assignment[other] == slot:
            return False
    return True

### Backtracking Solution

In [97]:
def backtracking(assignment):
    if len(assignment) == len(exams):
        return assignment.copy()

    exam = next(e for e in exams if e not in assignment)

    for slot in domains[exam]:
        if is_consistent(exam, slot, assignment):
            assignment[exam] = slot

            result = backtracking(assignment)
            if result is not None:
                return result

            del assignment[exam]

    return None

### Forward Checking to Remove Occupied Slots from Conflicting Exams

In [98]:
def forward_checking(assignment, remaining_domains):
    if len(assignment) == len(exams):
        return assignment.copy()

    exam = next(e for e in exams if e not in assignment)

    for slot in remaining_domains[exam]:
        if not is_consistent(exam, slot, assignment):
            continue

        assignment[exam] = slot

        new_domains = {
            e: values.copy()
            for e, values in remaining_domains.items()
        }

        new_domains[exam] = [slot]
        valid = True

        for other in conflicts[exam]:
            if other not in assignment:
                if slot in new_domains[other]:
                    new_domains[other].remove(slot)

                if not new_domains[other]:
                    valid = False
                    break

        if valid:
            result = forward_checking(
                assignment, new_domains
            )
            if result is not None:
                return result

        del assignment[exam]

    return None

In [99]:
def print_schedule(solution):
    if solution:
        for exam, slot in solution.items():
            print(f"{exam}: {slot}")
    else:
        print("No valid schedule exists.")

print("Backtracking Solution:")
bt_solution = backtracking({})
print_schedule(bt_solution)

print("\nForward Checking Solution:")
fc_solution = forward_checking({}, domains)
print_schedule(fc_solution)

Backtracking Solution:
AI: Monday 9 AM
DB: Monday 11 AM
OS: Monday 11 AM
CN: Monday 9 AM
PF: Monday 2 PM

Forward Checking Solution:
AI: Monday 9 AM
DB: Monday 11 AM
OS: Monday 11 AM
CN: Monday 9 AM
PF: Monday 2 PM


### Challenge: AI must be scheduled on Tuesday


In [100]:
challenge_domains = {
    e: values.copy() for e, values in domains.items()
}
challenge_domains["AI"] = ["Tuesday 9 AM"]

print("\nChallenge: AI Must Be on Tuesday")
challenge_solution = forward_checking(
    {}, challenge_domains
)
print_schedule(challenge_solution)


Challenge: AI Must Be on Tuesday
AI: Tuesday 9 AM
DB: Monday 9 AM
OS: Monday 9 AM
CN: Monday 11 AM
PF: Monday 2 PM


## Task 4 - N-Queens Problem

In [101]:
from collections import deque

def is_safe(col1, row1, col2, row2):
    return (
        row1 != row2 and
        abs(row1 - row2) != abs(col1 - col2)
    )

def revise(domains, xi, xj):
    revised = False

    for row_i in domains[xi][:]:
        if not any(
            is_safe(xi, row_i, xj, row_j)
            for row_j in domains[xj]
        ):
            domains[xi].remove(row_i)
            revised = True

    return revised

In [102]:
def ac3(domains):
    columns = list(domains.keys())

    queue = deque(
        (xi, xj)
        for xi in columns
        for xj in columns
        if xi != xj
    )

    while queue:
        xi, xj = queue.popleft()

        if revise(domains, xi, xj):
            if not domains[xi]:
                return False

            for xk in columns:
                if xk != xi and xk != xj:
                    queue.append((xk, xi))

    return True

In [103]:
def solve_nqueens(n, method="backtracking", use_mrv=False):
    columns = list(range(n))
    domains = {col: list(range(n)) for col in columns}

    attempts = 0
    backtracks = 0

    def backtrack(assignment, remaining_domains):
        nonlocal attempts, backtracks

        if len(assignment) == n:
            return assignment.copy()

        unassigned = [
            col for col in columns
            if col not in assignment
        ]

        # MRV heuristic with degree tie-breaker
        if use_mrv:
            col = min(
                unassigned,
                key=lambda c: (
                    len(remaining_domains[c]),
                    -(len(unassigned) - 1)
                )
            )
        else:
            col = unassigned[0]

        for row in remaining_domains[col]:
            attempts += 1

            # Check constraints with assigned queens
            if not all(
                is_safe(col, row, c, r)
                for c, r in assignment.items()
            ):
                continue

            assignment[col] = row

            new_domains = {
                c: values.copy()
                for c, values in remaining_domains.items()
            }
            new_domains[col] = [row]

            valid = True

            # Forward Checking
            if method == "forward":
                for other in unassigned:
                    if other == col:
                        continue

                    new_domains[other] = [
                        r for r in new_domains[other]
                        if is_safe(col, row, other, r)
                    ]

                    if not new_domains[other]:
                        valid = False
                        break

            # Maintaining Arc Consistency
            elif method == "ac3":
                valid = ac3(new_domains)

            if valid:
                result = backtrack(
                    assignment, new_domains
                )

                if result is not None:
                    return result

            del assignment[col]

        backtracks += 1
        return None

    # Apply AC-3 before starting search
    if method == "ac3" and not ac3(domains):
        return None, attempts, backtracks

    solution = backtrack({}, domains)

    return solution, attempts, backtracks

In [104]:
def print_board(solution, n):
    if solution is None:
        print("No solution exists.")
        return

    for row in range(n):
        for col in range(n):
            if solution[col] == row:
                print("Q", end=" ")
            else:
                print(".", end=" ")
        print()

In [105]:
for n in [4, 8, 10]:
    print(f"\n========== N = {n} ==========")

    for method in ["backtracking", "forward", "ac3"]:
        solution, attempts, backtracks = solve_nqueens(
            n, method
        )

        print(f"\nMethod: {method}")
        print("Solution:", solution)
        print("Attempted Assignments:", attempts)
        print("Backtracks:", backtracks)

        if solution:
            print("Chessboard:")
            print_board(solution, n)


========== N = 4 ==========

Method: backtracking
Solution: {0: 1, 1: 3, 2: 0, 3: 2}
Attempted Assignments: 26
Backtracks: 4
Chessboard:
. . Q . 
Q . . . 
. . . Q 
. Q . . 

Method: forward
Solution: {0: 1, 1: 3, 2: 0, 3: 2}
Attempted Assignments: 8
Backtracks: 2
Chessboard:
. . Q . 
Q . . . 
. . . Q 
. Q . . 

Method: ac3
Solution: {0: 1, 1: 3, 2: 0, 3: 2}
Attempted Assignments: 5
Backtracks: 0
Chessboard:
. . Q . 
Q . . . 
. . . Q 
. Q . . 

========== N = 8 ==========

Method: backtracking
Solution: {0: 0, 1: 4, 2: 7, 3: 5, 4: 2, 5: 6, 6: 1, 7: 3}
Attempted Assignments: 876
Backtracks: 105
Chessboard:
Q . . . . . . . 
. . . . . . Q . 
. . . . Q . . . 
. . . . . . . Q 
. Q . . . . . . 
. . . Q . . . . 
. . . . . Q . . 
. . Q . . . . . 

Method: forward
Solution: {0: 0, 1: 4, 2: 7, 3: 5, 4: 2, 5: 6, 6: 1, 7: 3}
Attempted Assignments: 88
Backtracks: 45
Chessboard:
Q . . . . . . . 
. . . . . . Q . 
. . . . Q . . . 
. . . . . . . Q 
. Q . . . . . . 
. . . Q . . . . 
. . . . . Q . . 
. .

In [106]:
print("\n========== MRV + DEGREE ==========")

solution, attempts, backtracks = solve_nqueens(
    8, method="forward", use_mrv=True
)

print("Solution:", solution)
print("Attempted Assignments:", attempts)
print("Backtracks:", backtracks)
print_board(solution, 8)


========== MRV + DEGREE ==========
Solution: {0: 0, 1: 4, 2: 7, 3: 5, 6: 1, 4: 2, 5: 6, 7: 3}
Attempted Assignments: 75
Backtracks: 44
Q . . . . . . . 
. . . . . . Q . 
. . . . Q . . . 
. . . . . . . Q 
. Q . . . . . . 
. . . Q . . . . 
. . . . . Q . . 
. . Q . . . . . 


## Task 5 - Resource Allocation Using CSP
### Task 5.1: Model the CSP

In [107]:
from collections import deque

projects = {
    "P1": {"students": 25, "equipment": "AI"},
    "P2": {"students": 18, "equipment": "Networking"},
    "P3": {"students": 20, "equipment": "General"},
    "P4": {"students": 15, "equipment": "AI"}
}

labs = {
    "A": {"capacity": 30, "equipment": {"AI"}},
    "B": {"capacity": 20, "equipment": {"Networking"}},
    "C": {"capacity": 25, "equipment": {"General"}}
}

variables = list(projects.keys())

def is_compatible(project, lab):
    return (
        projects[project]["students"] <= labs[lab]["capacity"]
        and projects[project]["equipment"] in labs[lab]["equipment"]
    )

domains = {
    p: [lab for lab in labs if is_compatible(p, lab)]
    for p in variables
}

def is_consistent(project, lab, assignment):
    return lab not in assignment.values()


### Task 5.2: Backtracking

In [108]:
# Task 5.2: Backtracking
def backtracking(assignment, stats):
    if len(assignment) == len(variables):
        return assignment.copy()

    project = next(
        p for p in variables if p not in assignment
    )

    for lab in domains[project]:
        stats["attempts"] += 1

        if is_consistent(project, lab, assignment):
            assignment[project] = lab

            result = backtracking(assignment, stats)

            if result is not None:
                return result

            del assignment[project]

    stats["backtracks"] += 1
    return None

### Task 5.3: Forward Checking


In [109]:
# Task 5.3: Forward Checking
def forward_checking(assignment, remaining_domains, stats):
    if len(assignment) == len(variables):
        return assignment.copy()

    project = next(
        p for p in variables if p not in assignment
    )

    for lab in remaining_domains[project]:
        stats["attempts"] += 1

        if not is_consistent(project, lab, assignment):
            continue

        assignment[project] = lab

        new_domains = {
            p: values.copy()
            for p, values in remaining_domains.items()
        }

        new_domains[project] = [lab]
        valid = True

        for other in variables:
            if other not in assignment:
                if lab in new_domains[other]:
                    new_domains[other].remove(lab)

                if not new_domains[other]:
                    valid = False
                    break

        if valid:
            result = forward_checking(
                assignment, new_domains, stats
            )

            if result is not None:
                return result

        del assignment[project]

    stats["backtracks"] += 1
    return None

### Task 5.4: Arc Consistency


In [110]:
def revise(domains, xi, xj):
    revised = False

    for lab_i in domains[xi][:]:
        if not any(
            lab_i != lab_j for lab_j in domains[xj]
        ):
            domains[xi].remove(lab_i)
            revised = True

    return revised


def ac3(domains):
    queue = deque(
        (xi, xj)
        for xi in variables
        for xj in variables
        if xi != xj
    )

    while queue:
        xi, xj = queue.popleft()

        if revise(domains, xi, xj):
            if not domains[xi]:
                return False

            for xk in variables:
                if xk != xi and xk != xj:
                    queue.append((xk, xi))

    return True


# Display solutions
def print_solution(solution):
    if solution:
        for project, lab in solution.items():
            print(f"{project} -> Lab {lab}")
    else:
        print("No valid allocation exists.")


print("Filtered Domains:")
for project, values in domains.items():
    print(f"{project}: {values}")

# Backtracking
bt_stats = {"attempts": 0, "backtracks": 0}
bt_solution = backtracking({}, bt_stats)

print("\nBacktracking Solution:")
print_solution(bt_solution)
print("Attempted Assignments:", bt_stats["attempts"])
print("Backtracks:", bt_stats["backtracks"])

# Forward Checking
fc_stats = {"attempts": 0, "backtracks": 0}
fc_solution = forward_checking({}, domains, fc_stats)

print("\nForward Checking Solution:")
print_solution(fc_solution)
print("Attempted Assignments:", fc_stats["attempts"])
print("Backtracks:", fc_stats["backtracks"])

# AC-3
ac3_domains = {
    p: values.copy() for p, values in domains.items()
}

print("\nDomains Before AC-3:")
for project, values in ac3_domains.items():
    print(f"{project}: {values}")

ac3_result = ac3(ac3_domains)

print("\nAC-3 Successful:", ac3_result)

print("\nDomains After AC-3:")
for project, values in ac3_domains.items():
    print(f"{project}: {values}")

# Backtracking after AC-3
print("\nAC-3 + Backtracking Solution:")

if ac3_result:
    ac3_stats = {"attempts": 0, "backtracks": 0}
    solution = backtracking({}, ac3_stats)
    print_solution(solution)
else:
    print("No valid allocation exists.")

Filtered Domains:
P1: ['A']
P2: ['B']
P3: ['C']
P4: ['A']

Backtracking Solution:
No valid allocation exists.
Attempted Assignments: 4
Backtracks: 4

Forward Checking Solution:
No valid allocation exists.
Attempted Assignments: 1
Backtracks: 1

Domains Before AC-3:
P1: ['A']
P2: ['B']
P3: ['C']
P4: ['A']

AC-3 Successful: False

Domains After AC-3:
P1: []
P2: ['B']
P3: ['C']
P4: ['A']

AC-3 + Backtracking Solution:
No valid allocation exists.


## Challenge Exercise

In [111]:
time_slots = ["9 AM", "11 AM"]

extended_domains = {
    project: [
        (lab, time)
        for lab in labs
        for time in time_slots
        if is_compatible(project, lab)
    ]
    for project in variables
}

def extended_backtracking(assignment, stats):
    if len(assignment) == len(variables):
        return assignment.copy()

    project = next(
        p for p in variables if p not in assignment
    )

    for lab_time in extended_domains[project]:
        stats["attempts"] += 1

        if lab_time not in assignment.values():
            assignment[project] = lab_time

            result = extended_backtracking(
                assignment, stats
            )

            if result is not None:
                return result

            del assignment[project]

    stats["backtracks"] += 1
    return None


stats = {"attempts": 0, "backtracks": 0}
solution = extended_backtracking({}, stats)

print("Resource Allocation With Time Slots:")

if solution:
    for project, (lab, time) in solution.items():
        print(f"{project} -> Lab {lab}, {time}")
else:
    print("No valid allocation exists.")

print("\nAttempted Assignments:", stats["attempts"])
print("Backtracks:", stats["backtracks"])


Resource Allocation With Time Slots:
P1 -> Lab A, 9 AM
P2 -> Lab B, 9 AM
P3 -> Lab C, 9 AM
P4 -> Lab A, 11 AM

Attempted Assignments: 5
Backtracks: 0
